# 3 · make — Helico structure-accuracy data (GDT-TS and lDDT)

Aggregates the per-target GDT-TS that [Open-Athena/helico](https://github.com/Open-Athena/helico)'s
exp14 published, over the same 333 FoldBench monomers #245 scores. Helico folds each protein from
a contact map; the arms differ only in where the contacts came from, so `Helico, no contacts` and
`Helico + true contacts` bracket what contact conditioning can do at all.

Restricted to targets **every** arm scored, so the arms are compared on one protein set rather
than on whichever ones each of them happened to finish.

CPU only; read anonymously from the public bucket.

In [ ]:
# Run from anywhere: figlib lives next to this notebook.
import sys
from pathlib import Path

HERE = Path.cwd() if (Path.cwd() / "figlib.py").exists() else Path("experiments/exp250_evals_exploration_notebook/figures")
sys.path.insert(0, str(HERE.resolve()))
import figlib

In [ ]:
# --- parameters -------------------------------------------------------------------------------
DATASET = "3_structure_accuracy"
# Two complementary structure metrics, both published per target. GDT-TS is the fraction of
# residues within a set of distance cutoffs after superposition — a global measure that a wrong
# domain orientation destroys. lDDT is superposition-free and local, so it credits locally
# correct geometry even when the global fold is off. Also available: tm_score, rmsd.
METRICS = ["gdt_ts", "lddt"]
BOOTSTRAP_DRAWS = 2000
BOOTSTRAP_SEED = 0
CLASSES = {"natural": 0, "designed": 1}   # name -> the `designed` flag it selects

# helico exp14's published `mf_L` arm conditions on #232's *sweep* checkpoint. #250 rescored the
# same 333 monomers with #232's better step-363000 checkpoint and re-ran that one arm, so this
# adds its rows from the repo. Everything else — off, oracle, both Protenix-v2 arms, ESMFold2 —
# is unchanged and still read from the published table, which is what keeps the comparison
# paired: same targets, same Helico checkpoint, same sampling, one input different.
EXTRA_ARM = {"path": "experiments/exp250_evals_exploration_notebook/data/helico_step363000/per_target.csv",
             "arm": "mf_L_363k"}
PARAMETERS = dict(metrics=METRICS, classes=list(CLASSES),
                  bootstrap_draws=BOOTSTRAP_DRAWS, bootstrap_seed=BOOTSTRAP_SEED,
                  extra_arm=EXTRA_ARM)
PARAMETERS

In [ ]:
import io

import pandas as pd

inputs = figlib.Inputs()
frame = pd.read_csv(io.BytesIO(inputs.fetch(f"{figlib.HELICO}/scores/per_target.csv")))

if EXTRA_ARM:
    extra_path = figlib.REPO / EXTRA_ARM["path"]
    if not extra_path.exists():
        raise SystemExit(f"{extra_path} is missing — it holds the re-run Helico arm's per-target "
                         "scores; see the experiment README for how it was produced")
    inputs.add_file(extra_path)
    extra = pd.read_csv(extra_path)
    extra = extra[extra.arm == EXTRA_ARM["arm"]]
    missing = [column for column in frame.columns if column not in extra.columns]
    if missing:
        raise SystemExit(f"the re-run arm is missing {missing}; it has to carry the same columns "
                         "as the published table or the two cannot be pooled")
    frame = pd.concat([frame, extra[frame.columns]], ignore_index=True)
    print(f"added {EXTRA_ARM['arm']}: {len(extra)} rows")

scored = frame[frame.status == "ok"]
complete = scored.groupby("target_id").arm.nunique()
keep = set(complete[complete == scored.arm.nunique()].index)
dropped = sorted(set(scored.target_id) - keep)
per_target = scored[scored.target_id.isin(keep)]
print(f"{scored.arm.nunique()} arms · {len(keep)} targets scored by all of them"
      + (f" · {len(dropped)} dropped: {dropped}" if dropped else ""))

In [ ]:
rows = []
for metric in METRICS:
    for class_name, designed in CLASSES.items():
        subset = per_target[per_target.designed == designed]
        for arm, group in subset.groupby("arm"):
            mean, low, high = figlib.bootstrap_mean(group[metric].values, BOOTSTRAP_DRAWS,
                                                    BOOTSTRAP_SEED)
            rows.append(dict(metric=metric, protein_class=class_name, arm=arm, n=len(group),
                             value=mean, ci_low=low, ci_high=high))

summary = pd.DataFrame(rows).sort_values(["metric", "protein_class", "value"],
                                         ascending=[True, True, False])
for metric in METRICS:
    print(f"--- {metric} ---")
    print(summary[summary.metric == metric].to_string(index=False,
                                                      float_format=lambda v: f"{v:.3f}"))

In [ ]:
figlib.write_dataset(
    DATASET,
    notebook="3_make_structure_accuracy_data.ipynb",
    parameters=PARAMETERS,
    inputs=inputs,
    files={
        "summary.csv": lambda path: summary.to_csv(path, index=False),
        "per_target.csv": lambda path: per_target.to_csv(path, index=False),
    },
    extra={
        "metrics": {"names": METRICS, "source": "Open-Athena/helico exp14"},
        "arms_dropped_targets": dropped,
        "arms": sorted(per_target.arm.unique()),
    })